# Denoising Shabby Pages

Project References

https://www.kaggle.com/competitions/denoising-shabby-pages

https://wandb.ai/dbrambilla13/shabby-pages?workspace=user-dbrambilla13

https://github.com/dbrambilla13/denoising-shabby-pages

Import Dataset

In [ ]:
from data import ShabbyPagesDataModule
data_module = ShabbyPagesDataModule(
    batch_size=16,
    num_workers=7,
)

In [ ]:
# !pip install wandb 
import wandb

wandb.login()


Setup training

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import WandbLogger
from lightning import Trainer


early_stopping = EarlyStopping(
    # monitor='valid_mse',
    monitor="mse_val",
    patience=25,
    mode="min",
)

checkpoint_callback = ModelCheckpoint(
    dirpath="saved_models/",
    monitor="mse_val",
    mode="min",
    filename="{epoch}-{mse_val:.3f}",
    auto_insert_metric_name=True,
)

wandb_logger = WandbLogger(project="insert_project_name_here")


trainer = Trainer(
    # max_epochs=1,
    max_epochs=1,  # 500 should be a good number, here is 1 only for didactit purposes
    accelerator="auto",  # This can be changed to auto for automatic device selection
    precision="16-mixed",
    logger=wandb_logger,
    callbacks=[
        early_stopping,
        checkpoint_callback,
    ],
)

In [ ]:
from model import DenoisingNet

model = DenoisingNet()

Let's train the model!

In [ ]:

trainer.fit(model=model, datamodule=data_module)

And now let's test it

In [ ]:
trained_model = DenoisingNet.load_from_checkpoint("saved_models/epoch=0-mse_val=0.099.ckpt").eval().cpu()

In [ ]:
trainer.test(
    model=trained_model,
    datamodule=data_module,
)